# Day-3 Practice Tasks

Practice exercises spanning the Day-3 topics: data serialization, HTTP requests, MySQL database
connectivity, and Flask APIs.

- Task 1 (Easy): JSON & Pickle serialization
- Task 2 (Easy): HTTP requests with `requests`
- Task 3 (Medium): MySQL CRUD & transactions
- Task 4 (Medium): Flask API

## Task 1 (Easy): JSON & Pickle Serialization

Write two helper functions, `save_customer_json(customer, path)` and
`load_customer_json(path)`, that serialize/deserialize a customer `dict` to/from a JSON file.

Then add a `joined` field to the customer holding a `datetime` object and try to save it with
`save_customer_json` — notice it raises `TypeError` because `datetime` isn't JSON-serializable.
Save that version with `pickle` instead and load it back, confirming the `datetime` survives the
round trip.

(Uses a temporary directory so the notebook does not leave files behind.)

In [1]:
# Solution
import os
import json
import pickle
import tempfile
from datetime import datetime

def save_customer_json(customer, path):
    with open(path, "w") as f:
        json.dump(customer, f, indent=2)

def load_customer_json(path):
    with open(path, "r") as f:
        return json.load(f)

demo_dir = tempfile.mkdtemp(prefix="day3_task1_")
json_path = os.path.join(demo_dir, "customer.json")
pickle_path = os.path.join(demo_dir, "customer.pkl")

customer = {"id": 1, "name": "Ben", "address": "Park Lane 38", "active": True}

save_customer_json(customer, json_path)
print("Loaded from JSON:", load_customer_json(json_path))

customer["joined"] = datetime.now()
try:
    save_customer_json(customer, json_path)
except TypeError as e:
    print("JSON failed as expected:", e)

with open(pickle_path, "wb") as f:
    pickle.dump(customer, f)

with open(pickle_path, "rb") as f:
    restored = pickle.load(f)

print("Restored from pickle:", restored)
print("joined type:", type(restored["joined"]))

os.remove(json_path)
os.remove(pickle_path)
os.rmdir(demo_dir)
print("Cleaned up temporary files and directory")

Loaded from JSON: {'id': 1, 'name': 'Ben', 'address': 'Park Lane 38', 'active': True}
JSON failed as expected: Object of type datetime is not JSON serializable
Restored from pickle: {'id': 1, 'name': 'Ben', 'address': 'Park Lane 38', 'active': True, 'joined': datetime.datetime(2026, 8, 20, 7, 57, 6, 972564)}
joined type: <class 'datetime.datetime'>
Cleaned up temporary files and directory


## Task 2 (Easy): HTTP Requests with `requests`

Send a GET request to `https://postman-echo.com/get` with query params `{'name': 'gautham',
'course': 'python'}` and print the status code and the echoed `args` from the JSON response.

Then request `https://postman-echo.com/status/404`, call `raise_for_status()` inside a
try/except, and print the caught `requests.exceptions.HTTPError`.

In [2]:
# Solution
import requests

response = requests.get("https://postman-echo.com/get", params={"name": "gautham", "course": "python"})
print("Status code:", response.status_code)
print("Echoed args:", response.json()["args"])

try:
    r = requests.get("https://postman-echo.com/status/404")
    r.raise_for_status()
except requests.exceptions.HTTPError as e:
    print("Caught error:", e)

Status code: 200
Echoed args: {'name': 'gautham', 'course': 'python'}


Caught error: 404 Client Error: Not Found for url: https://postman-echo.com/status/404


## Task 3 (Medium): MySQL CRUD & Transactions

Using `mysql.connector` (same connection pattern as `mysql.ipynb`):

1. Connect to a local MySQL server and create a `products` table with columns
   `id INT AUTO_INCREMENT PRIMARY KEY`, `name VARCHAR(255)`, `price FLOAT`.
2. Insert several rows in one batch using `executemany`.
3. Query all products ordered by `price` descending, and query products with `price > 50`.
4. Update one product's price and delete another, committing after each successful change.
5. Wrap a deliberately invalid statement (e.g. inserting into a nonexistent column) in a
   try/except that calls `mydb.rollback()` on failure instead of leaving the transaction open.

**Note:** this task needs `pip install mysql-connector-python` and a running local MySQL server
(as used in `mysql.ipynb`); neither is available in this environment, so the cell below has
correct code but was not executed here.

In [ ]:
# Solution (not executed in this environment - requires mysql-connector-python + a running MySQL server)
import mysql.connector

mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="shop"
)
mycursor = mydb.cursor()

mycursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(255),
        price FLOAT
    )
""")

products = [
    ("Keyboard", 45.0),
    ("Monitor", 150.0),
    ("Mouse", 20.0),
    ("Webcam", 60.0),
]
mycursor.executemany("INSERT INTO products (name, price) VALUES (%s, %s)", products)
mydb.commit()
print(mycursor.rowcount, "records inserted")

mycursor.execute("SELECT * FROM products ORDER BY price DESC")
print("All products by price:", mycursor.fetchall())

mycursor.execute("SELECT * FROM products WHERE price > %s", (50,))
print("Products over $50:", mycursor.fetchall())

mycursor.execute("UPDATE products SET price = %s WHERE name = %s", (55.0, "Keyboard"))
mydb.commit()
print(mycursor.rowcount, "record(s) updated")

mycursor.execute("DELETE FROM products WHERE name = %s", ("Mouse",))
mydb.commit()
print(mycursor.rowcount, "record(s) deleted")

try:
    mycursor.execute("INSERT INTO products (name, price, nonexistent_column) VALUES (%s, %s, %s)",
                      ("Bad Product", 10.0, "oops"))
    mydb.commit()
except mysql.connector.Error as e:
    mydb.rollback()
    print("Transaction rolled back after error:", e)

## Task 4 (Medium): Flask API

Build a small Flask app with an in-memory list of items exposing:
- `GET /items` — list all items as JSON
- `GET /items/<id>` — return one item by id, or a 404 JSON error if not found
- `POST /items` — add a new item from a JSON body and return it with a 201 status

Exercise all three routes using Flask's `app.test_client()` (no server needs to be started).

In [3]:
# Solution
from flask import Flask, jsonify, request

app = Flask(__name__)
items = [
    {"id": 1, "name": "Widget"},
    {"id": 2, "name": "Gadget"},
]

@app.get("/items")
def list_items():
    return jsonify(items)

@app.get("/items/<int:item_id>")
def get_item(item_id):
    for item in items:
        if item["id"] == item_id:
            return jsonify(item)
    return jsonify({"error": "not found"}), 404

@app.post("/items")
def create_item():
    new_item = request.get_json()
    new_item["id"] = max(i["id"] for i in items) + 1
    items.append(new_item)
    return jsonify(new_item), 201

client = app.test_client()

resp = client.get("/items")
print(resp.status_code, resp.get_json())

resp = client.get("/items/1")
print(resp.status_code, resp.get_json())

resp = client.get("/items/99")
print(resp.status_code, resp.get_json())

resp = client.post("/items", json={"name": "Gizmo"})
print(resp.status_code, resp.get_json())

resp = client.get("/items")
print(resp.status_code, resp.get_json())

200 [{'id': 1, 'name': 'Widget'}, {'id': 2, 'name': 'Gadget'}]
200 {'id': 1, 'name': 'Widget'}
404 {'error': 'not found'}
201 {'id': 3, 'name': 'Gizmo'}
200 [{'id': 1, 'name': 'Widget'}, {'id': 2, 'name': 'Gadget'}, {'id': 3, 'name': 'Gizmo'}]
